# Tutorial: Training and Deploying a Streaming Fraud-Detection Pipeline on Google Cloud

This tutorial walks through a single end-to-end portfolio project in **two parts**:

- **Part 1 — Train.** Provision a private cloud environment that builds a Docker image, trains a PySpark fraud-detection model on Google Cloud, and pushes a versioned scoring image to a private container registry.
- **Part 2 — Stream.** Provision a second, separate environment that pulls that scoring image, replays credit-card transactions through Google Cloud Pub/Sub, scores them in near-real time, and writes a fraud report.

Both parts are driven by **Terraform** (infrastructure as code), **Docker** (reproducible runtime), and the **GCE metadata server** (keyless cloud-native authentication). After teardown, **zero cloud state remains.**

> **A note on style.** The repository's source files (`*.tf`, `*.py`, `Dockerfile`, `startup.sh`) are commented inline for readability — they explain *how* each piece works. This notebook explains *why* each piece exists and how the pieces fit together. Where a snippet is shown inline below, it is the headline; the full file is always linked.

## Table of contents

**Intro**
- [1.1 The problem](#11-the-problem)
- [1.2 Why cloud, IaC, and containers](#12-why-cloud-iac-and-containers)
- [1.3 Why three Terraform stacks](#13-why-three-terraform-stacks)
- [1.4 Toolchain](#14-toolchain)
- [1.5 Prerequisites](#15-prerequisites)
- [1.6 End-to-end lifecycle](#16-end-to-end-lifecycle)

**Part 1 — Train and publish the scoring image**
- [2.1 Goal of Part 1](#21-goal-of-part-1)
- [2.2 The data stack — shared, long-lived resources](#22-the-data-stack-shared-long-lived-resources)
- [2.3 The train stack — private network and VM](#23-the-train-stack-private-network-and-vm)
- [2.4 How scripts get onto the VM (the templatefile + metadata-token trick)](#24-how-scripts-get-onto-the-vm)
- [2.5 Image layering — base image, then derived scoring image](#25-image-layering-base-image-then-derived-scoring-image)
- [2.6 The PySpark training payload](#26-the-pyspark-training-payload)
- [2.7 Running it](#27-running-it)
- [2.8 Inspecting outputs](#28-inspecting-outputs)
- [2.9 Vulnerability scanning](#29-vulnerability-scanning)
- [2.10 Tearing down the train compute](#210-tearing-down-the-train-compute)

**Part 2 — Deploy and score the stream**
- [3.1 Goal of Part 2](#31-goal-of-part-2)
- [3.2 Cross-stack data sharing](#32-cross-stack-data-sharing)
- [3.3 The stream stack — Pub/Sub and the consumer VM](#33-the-stream-stack-pubsub-and-the-consumer-vm)
- [3.4 The publisher and the consumer](#34-the-publisher-and-the-consumer)
- [3.5 Running it and the sample report](#35-running-it-and-the-sample-report)
- [3.6 Tearing everything down](#36-tearing-everything-down)

**Recap**
- [4.1 Concepts → where they appear](#41-concepts-where-they-appear)
- [4.2 Applying this to a real fraud-detection workload](#42-applying-this-to-a-real-fraud-detection-workload)
- [4.3 Where to go next](#43-where-to-go-next)

# Intro

!!ADD HIGH-LEVEL ARCHITECTURE DIAGRAM HERE — THREE TERRAFORM STACKS (data / train / stream), THE SHARED GCS BUCKET + ARTIFACT REGISTRY, THE TWO PRIVATE VMs, THE PUB/SUB TOPIC, AND ARROWS SHOWING DATA FLOW (LAPTOP → BUCKET → TRAIN VM → REGISTRY → STREAM VM → REPORT BACK TO BUCKET)!!

## 1.1 The problem

Credit-card fraud detection is a textbook commercial application of cloud computing. The dataset is **highly imbalanced** (genuine fraud is ~0.17% of transactions), the workload has **two very different shapes** — a long batch *training* job and a continuous *inference* stream — and the cost of getting it wrong (missed fraud, or worse, false positives that block legitimate cardholders) makes reproducibility and auditability non-negotiable.

We use the public **Kaggle ULB credit-card dataset** (284,807 transactions, 30 numeric predictors, binary `Class` label). The model itself is intentionally simple — a logistic regression in a `pyspark.ml.Pipeline` — because the goal of the portfolio is to demonstrate the **cloud architecture** around the model, not the model itself.

## 1.2 Why cloud, IaC, and containers

Three problems show up the moment you try to run a PySpark job somewhere other than your laptop:

1. **Runtime drift.** PySpark needs a specific Java version. Ubuntu 22.04 ships one, your laptop has another, the next CI runner has a third. Containerising fixes the runtime — the *image* becomes the artifact, not the script.
2. **Environment drift.** A VM that was hand-clicked into existence cannot be recreated identically. **Infrastructure as Code** (Terraform) makes the entire cloud environment a versioned text file: `terraform apply` builds it, `terraform destroy` removes it.
3. **Identity and secrets.** A real cloud workload needs to read data from one place and write results to another, without shipping passwords. We use the **GCE metadata server** (covered in §2.4) so the VM proves its identity simply by *being itself* — no key files ever touch the VM.

Together, these three ideas are what makes the pipeline reproducible: anyone with `gcloud` and `terraform` installed can clone the repo and run the same workload in their own GCP project.

## 1.3 Why three Terraform stacks

The infrastructure is split into three independent Terraform stacks under [infra/terraform/](infra/terraform/):

| Stack | Lifetime | Contents |
|---|---|---|
| **`data/`** | Long-lived (survives between Part 1 and Part 2) | GCS bucket, dataset upload, Artifact Registry, shared service account, IAM bindings |
| **`train/`** | Short-lived (only during Part 1) | Private VPC, IAP-only SSH, Cloud NAT, train VM with startup script |
| **`stream/`** | Short-lived (only during Part 2) | Private VPC, IAP-only SSH, Cloud NAT, Pub/Sub topic + subscription, stream VM |

The split matters because **compute is expensive and ephemeral**, while **data and registries are cheap and stateful**. Destroying the train VM the moment training finishes (Part 1 ends) costs nothing but saves money; the *trained image* it pushed survives in the registry for Part 2 to consume. The same pattern applies in production: short-lived training clusters, long-lived model registries.

The three stacks talk to each other through Terraform's `remote_state` data source — see [§3.2](#32-cross-stack-data-sharing) for the snippet.

## 1.4 Toolchain

- **Terraform** (≥ 1.5) — declarative provisioning of every cloud resource.
- **Docker** — image build/run on the VM (not on your laptop).
- **Google Cloud Platform** services: **Compute Engine** (VMs), **Cloud Storage** (dataset + reports), **Artifact Registry** (private Docker images), **Pub/Sub** (streaming transport), **IAP** (zero-public-IP SSH), **Cloud NAT** (egress for private VMs), **Container Analysis** (vulnerability scanning).
- **PySpark 3.5** — `Pipeline`, `LogisticRegression`, `BinaryClassificationEvaluator`, and `PipelineModel.save/load` for model persistence between Part 1 and Part 2.
- **Python `google-cloud-pubsub`** — publisher in Part 2.

## 1.5 Prerequisites

- A GCP project with billing enabled and your `gcloud` CLI authenticated:
  ```bash
  gcloud auth login
  gcloud auth application-default login
  gcloud config set project <project_id>
  ```
- Terraform `>= 1.5` on `PATH`.
- The Kaggle [creditcard.csv](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) dataset placed at `data/creditcard.csv`. The file is gitignored; Terraform uploads it to GCS during Part 1.
- Container Analysis enabled once per project (free; only needed for §2.9):
  ```bash
  gcloud services enable containerscanning.googleapis.com --project <project_id>
  ```
- Set `project_id` (and optionally `region`) in [infra/terraform/data/terraform.tfvars](infra/terraform/data/terraform.tfvars) before the first apply.

## 1.6 End-to-end lifecycle

From a clean project, six commands take you all the way through and back:

```bash
1. cd infra/terraform/data   && terraform init && terraform apply   # bucket + registry + SA
2. cd ../train               && terraform init && terraform apply   # train VM trains model + pushes scoring image
3. cd ../train               && terraform destroy                    # train compute gone, registry image stays
4. cd ../stream              && terraform init && terraform apply   # stream VM pulls image, scores Pub/Sub, writes report
5. cd ../stream              && terraform destroy                    # stream compute gone
6. cd ../data                && terraform destroy                    # bucket + registry gone — zero cloud state
```

Or run [scripts/destroy_everything.sh](scripts/destroy_everything.sh) (Linux/Mac) or [scripts/destroy_everything.ps1](scripts/destroy_everything.ps1) (Windows) at the end to do steps 3+5+6 in one shot.

# Part 1 — Train and publish the scoring image

!!ADD TRAIN PIPELINE DIAGRAM HERE — LAPTOP RUNS `terraform apply` → TRAIN VM BOOTS → STARTUP SCRIPT BUILDS BASE IMAGE → RUNS PYSPARK TRAINING IN CONTAINER → SAVES PIPELINEMODEL → BUILDS DERIVED `fraud-scoring:v5` IMAGE → PUSHES TO ARTIFACT REGISTRY → UPLOADS `summary.json` TO GCS!!

## 2.1 Goal of Part 1

By the end of Part 1 the following exist in your GCP project:

- A GCS bucket holding the dataset and a `summary.json` of training metrics.
- A private Artifact Registry repository containing two image tags:
  - `fraud-detection:base` — Python + Java + PySpark + scripts (no model).
  - `fraud-scoring:v5` — base + the trained `PipelineModel` baked in.
- The train VM is **gone** (we destroy it in §2.10). The scoring image survives in the registry, ready for Part 2.

## 2.2 The data stack — shared, long-lived resources

[infra/terraform/data/main.tf](infra/terraform/data/main.tf) provisions everything that should *outlive* a single training run:

- A `google_storage_bucket` named `<project_id>-fraud-dataset` (with a 7-day object lifecycle as a billing safety net).
- Two `google_storage_bucket_object` uploads — the full training CSV and a smaller `transactions_stream.csv` for Part 2.
- A `google_artifact_registry_repository` for Docker images.
- A single shared `google_service_account` used by both the train and stream VMs.

The interesting part is the **scoped IAM bindings** — least privilege in HCL form:

```hcl
# Bucket-scoped: SA gets object-level read/write on THIS bucket only.
resource "google_storage_bucket_iam_member" "dataset_admin" {
  bucket = google_storage_bucket.dataset.name
  role   = "roles/storage.objectAdmin"
  member = "serviceAccount:${google_service_account.fraud_vm_sa.email}"
}

# Repository-scoped: SA gets push/pull on THIS registry only.
resource "google_artifact_registry_repository_iam_member" "image_writer" {
  location   = google_artifact_registry_repository.images.location
  repository = google_artifact_registry_repository.images.name
  role       = "roles/artifactregistry.writer"
  member     = "serviceAccount:${google_service_account.fraud_vm_sa.email}"
}
```

Each binding attaches a role to a *specific resource*, not the whole project. If the train VM is ever compromised, the blast radius is one bucket and one registry — not the entire GCP project.

## 2.3 The train stack — private network and VM

[infra/terraform/train/main.tf](infra/terraform/train/main.tf) builds a self-contained network around the train VM. The VM **has no public IP**: outbound internet goes through Cloud NAT, inbound SSH goes through Identity-Aware Proxy (IAP).

!!ADD NETWORK DIAGRAM HERE — PRIVATE SUBNET WITH THE TRAIN VM (NO PUBLIC IP) IN THE MIDDLE; ON THE LEFT, IAP TUNNEL FROM THE OPERATOR'S LAPTOP REACHING IN VIA `35.235.240.0/20`; ON THE RIGHT, CLOUD NAT GATEWAY CARRYING OUTBOUND TRAFFIC TO THE PUBLIC INTERNET (FOR `apt`, `docker pull`, `gcloud`)!!

**SSH from IAP only** (defence in depth — slide Doc 1):

```hcl
resource "google_compute_firewall" "allow_iap_ssh" {
  name    = "${var.instance_name}-allow-iap-ssh"
  network = google_compute_network.train_vpc.name

  allow {
    protocol = "tcp"
    ports    = ["22"]
  }

  source_ranges = ["35.235.240.0/20"]   # IAP's address range — nothing else can reach :22
  target_tags   = ["ssh-iap"]
}
```

**Outbound through Cloud NAT** so the private VM can `apt-get update` and pull the Python base image:

```hcl
resource "google_compute_router" "train_router" {
  name    = "${var.vpc_name}-router"
  region  = local.region
  network = google_compute_network.train_vpc.id
}

resource "google_compute_router_nat" "train_nat" {
  name                               = "${var.vpc_name}-nat"
  router                             = google_compute_router.train_router.name
  region                             = local.region
  nat_ip_allocate_option             = "AUTO_ONLY"
  source_subnetwork_ip_ranges_to_nat = "ALL_SUBNETWORKS_ALL_IP_RANGES"
}
```

The VM itself attaches the shared service account from §2.2 with the broad `cloud-platform` OAuth scope — the *actual* permissions are constrained by the resource-scoped IAM bindings, not the scope.

## 2.4 How scripts get onto the VM

The train VM never clones the repo. Instead, Terraform reads every script and Dockerfile from your laptop's filesystem and bakes them into the VM's startup script using **`templatefile()`**:

```hcl
metadata_startup_script = templatefile("${path.module}/startup.sh", {
  dockerfile_txt         = file("${path.module}/../../../docker/Dockerfile")
  dockerfile_scoring_txt = file("${path.module}/../../../docker/Dockerfile.scoring")
  requirements_txt       = file("${path.module}/../../../requirements.txt")
  fraud_script_py        = file("${path.module}/../../../scripts/fraud_detection_pyspark.py")
  score_script_py        = file("${path.module}/../../../scripts/score_stream.py")
  publish_script_py      = file("${path.module}/../../../scripts/publish_transactions.py")
  # ... plus the bucket name, region, image URLs, etc.
})
```

Inside [infra/terraform/train/startup.sh](infra/terraform/train/startup.sh) the placeholders look like `${dockerfile_txt}` and Terraform substitutes the file contents at apply time. The VM boots with everything it needs already inside its cloud-init metadata.

**The cloud-native authentication idiom.** The VM still needs to *download* the dataset from GCS and *push* the scoring image to Artifact Registry. It does that without any key files, using the GCE metadata server:

```bash
ACCESS_TOKEN="$(curl --fail --silent --show-error \
  -H 'Metadata-Flavor: Google' \
  'http://metadata.google.internal/computeMetadata/v1/instance/service-accounts/default/token' \
  | jq -r '.access_token')"

curl --fail --location \
  -H "Authorization: Bearer $ACCESS_TOKEN" \
  -o "$WORKDIR/data/creditcard.csv" \
  "https://storage.googleapis.com/storage/v1/b/$DATA_BUCKET/o/$ENCODED_OBJECT?alt=media"
```

The address `metadata.google.internal` resolves to `169.254.169.254` — a magic IP that the GCE hypervisor intercepts. Only code running *inside* a GCE VM can reach it, and the required `Metadata-Flavor: Google` header proves the request is intentional. The token returned is tied to the VM's attached service account; no credential file ever leaves Google. The same pattern works on AWS and Azure with different endpoints — it's the **portable cloud-native auth idiom**.

## 2.5 Image layering — base image, then derived scoring image

Two Dockerfiles, one shared layer cache. The base image is built once, used for training, and reused as the parent of the scoring image — the trained model is the only thing copied in.

!!ADD IMAGE-LAYERING DIAGRAM HERE — LEFT: BASE IMAGE LAYERS (python:3.12-slim → openjdk-21 → pip install requirements → COPY scripts). RIGHT: SCORING IMAGE = BASE + ONE NEW LAYER (`COPY model/`). ARROW SHOWING SCORING IMAGE INHERITS EVERYTHING TO THE LEFT!!

**Base image** — [docker/Dockerfile](docker/Dockerfile):

```dockerfile
FROM python:3.12-slim
WORKDIR /app

RUN apt-get update && \
    apt-get install -y --no-install-recommends openjdk-21-jre-headless && \
    rm -rf /var/lib/apt/lists/*

ENV JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64
ENV SPARK_LOCAL_HOSTNAME=localhost

COPY requirements.txt .
RUN pip install --no-cache-dir --upgrade pip && \
    pip install --no-cache-dir -r requirements.txt

COPY scripts ./scripts
RUN mkdir -p data output

CMD ["python", "scripts/fraud_detection_pyspark.py", "--data-path", "data/creditcard.csv", "--output-dir", "output", "--max-iter", "10"]
```

Three things worth knowing:
1. `requirements.txt` is copied **before** `scripts/`. If you only edit a Python script, Docker reuses the cached `pip install` layer and rebuilds in seconds.
2. `--no-install-recommends` and the `apt` cleanup keep the image small.
3. Pinning `JAVA_HOME` and `SPARK_LOCAL_HOSTNAME` removes two of the most common PySpark-in-a-container surprises.

**Derived scoring image** — [docker/Dockerfile.scoring](docker/Dockerfile.scoring):

```dockerfile
ARG BASE_IMAGE
FROM ${BASE_IMAGE}

COPY model /app/model

CMD ["python", "scripts/score_stream.py", "--model-dir", "/app/model"]
```

That's the entire derived image: take everything from base, add one folder, override the default command. Because the scoring image only adds a single layer on top of the cached base, the second `docker build` and the `docker push` to Artifact Registry are both fast — only the new layer is uploaded.

Full `requirements.txt` (with the `pyspark==3.5.3` pin and explicit `numpy`) lives at [requirements.txt](requirements.txt).

## 2.6 The PySpark training payload

The training script is intentionally a small `Pipeline`. The headline is the assembly itself — three stages glued together so the *exact same transformations* can be reused at inference time in Part 2:

```python
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
scaler    = StandardScaler(inputCol="raw_features", outputCol="features",
                           withMean=False, withStd=True)
lr        = LogisticRegression(featuresCol="features", labelCol="label",
                               weightCol="weight",
                               maxIter=args.max_iter,
                               threshold=args.decision_threshold)

pipeline = Pipeline(stages=[assembler, scaler, lr])
model    = pipeline.fit(train_df)

model.write().overwrite().save(args.model_output_dir)   # <- the artifact Part 2 will load
```

Two evaluators run after training; AUC-PR is the honest one for imbalanced data:

```python
auc    = BinaryClassificationEvaluator(metricName="areaUnderROC").evaluate(predictions)
auc_pr = BinaryClassificationEvaluator(metricName="areaUnderPR").evaluate(predictions)
```

On the cloud run the model achieved **AUC-ROC ≈ 0.96** and **AUC-PR ≈ 0.71** — see §3.5 for the full numbers. The full script lives at [scripts/fraud_detection_pyspark.py](scripts/fraud_detection_pyspark.py).

## 2.7 Running it

From the repo root:

```bash
cd infra/terraform/data
terraform init
terraform apply        # type 'yes' — creates bucket + uploads dataset + creates registry + SA + IAM (~2 min)

cd ../train
terraform init
terraform apply        # type 'yes' — creates VPC/NAT/IAP + train VM (~1 min for TF; then VM runs ~5–8 min)
```

Terraform finishes once the VM is created. The startup script then runs *on the VM*, asynchronously: install Docker → download the dataset from GCS → build the base image → train inside the container → save `PipelineModel` → build the derived scoring image → push to Artifact Registry → upload `summary.json` to GCS.

## 2.8 Inspecting outputs

The README's [Inspecting each phase](README.md#inspecting-each-phase) section lists the full set of `gcloud` commands. The two most useful for Part 1:

```bash
# Watch the VM's startup log live (build → train → push)
gcloud compute ssh fraud-train-vm --zone europe-west2-a --tunnel-through-iap \
  --command 'sudo tail -f /var/log/startup.log'

# Pretty-print training metrics straight from GCS — no SSH
gcloud storage cat gs://<bucket>/summaries/summary.json | jq
```

The first SSH happens through IAP because the VM has no public IP — `--tunnel-through-iap` is what makes `gcloud ssh` work despite the firewall locking port 22 to `35.235.240.0/20`.

## 2.9 Vulnerability scanning

Once `containerscanning.googleapis.com` is enabled on the project, **every image pushed to Artifact Registry is auto-scanned** by Container Analysis. Inspect the `fraud-scoring:v5` findings with:

```bash
gcloud artifacts docker images list \
  europe-west2-docker.pkg.dev/<project_id>/fraud-detection-images/fraud-scoring \
  --show-occurrences --occurrence-filter='kind="VULNERABILITY"'
```

On our run this returned **2 CRITICAL, 32 HIGH, 35 MEDIUM, 28 LOW, 26 MINIMAL** findings — all in the Debian trixie base layer (`glibc`, `tar`, `nss`, `sqlite3`, `shadow`, `systemd`), none in application code or Python dependencies. Two flags on every finding make them non-actionable today:

- `effectiveSeverity: MINIMAL` — Google's contextual re-score: the vulnerable code paths are not reachable in this image's runtime context.
- `fixedVersion.kind: MAXIMUM` — no upstream fix exists in Debian, so `apt-get upgrade` would change nothing.

Real mitigations are structural, not patch-based: switch to a **distroless** or **Chainguard** base, **pin the base by digest** (`FROM python:3.12-slim@sha256:...`) so re-scans run against a known image, and **drop the JRE from the scoring image** (only training needs Spark).

## 2.10 Tearing down the train compute

The instant training is finished and the scoring image is in the registry, the train VM has nothing left to do. **Destroy it now** so you stop paying for compute you don't need:

```bash
cd infra/terraform/train
terraform destroy
```

The data stack (bucket + registry + scoring image + SA) **survives** this teardown. Part 2 will read the scoring image from the registry and the streaming subset from the bucket.

# Part 2 — Deploy and score the stream

!!ADD STREAM PIPELINE DIAGRAM HERE — LAPTOP RUNS `terraform apply` → STREAM VM BOOTS → DOCKER PULL `fraud-scoring:v5` FROM REGISTRY → CONSUMER CONTAINER STARTS LISTENING ON PUB/SUB SUBSCRIPTION → PUBLISHER (HOST PROCESS, NOT CONTAINERISED) READS `transactions_stream.csv` → PUBLISHES JSON MESSAGES TO PUB/SUB TOPIC AT 25 MSG/S → CONSUMER PULLS, BATCHES, SCORES THROUGH `PipelineModel` → WRITES `fraud_report.json` → UPLOADS TO GCS!!

## 3.1 Goal of Part 2

Part 2 demonstrates that a model trained inside one short-lived environment can be **deployed and consumed by a totally separate environment** without manual artefact handoff. By the end:

- A new private VM has booted, pulled `fraud-scoring:v5` from Artifact Registry, and is running the consumer in Docker.
- A publisher has replayed 3,000 transactions through Pub/Sub at ~25 msg/s.
- A `fraud_report.json` listing every flagged transaction has been written and uploaded to GCS.
- The stream VM is destroyed; the report persists in the bucket.

## 3.2 Cross-stack data sharing

The stream stack needs the bucket name, registry URL, and service-account email created by the data stack — *without* hardcoding them. Terraform's `terraform_remote_state` data source reads the data stack's state file and exposes its outputs:

```hcl
data "terraform_remote_state" "data" {
  backend = "local"
  config = {
    path = "${path.module}/../data/terraform.tfstate"
  }
}

locals {
  project_id            = data.terraform_remote_state.data.outputs.project_id
  dataset_bucket_name   = data.terraform_remote_state.data.outputs.dataset_bucket_name
  artifact_repo_url     = data.terraform_remote_state.data.outputs.artifact_repo_url
  service_account_email = data.terraform_remote_state.data.outputs.service_account_email
  scoring_image_url     = "${local.artifact_repo_url}/fraud-scoring:v5"
}
```

If you change the bucket name in the data stack and re-apply, the stream stack picks up the new value automatically on its next `terraform plan`. In a multi-team setup, the same pattern works with a **remote backend** (GCS bucket holding the state file) so different teams can read each other's outputs without sharing local files.

## 3.3 The stream stack — Pub/Sub and the consumer VM

[infra/terraform/stream/main.tf](infra/terraform/stream/main.tf) reuses the same private-VPC + IAP + Cloud NAT pattern from §2.3 (the snippets are nearly identical) and adds the messaging plumbing:

```hcl
resource "google_pubsub_topic" "transactions" {
  name = var.pubsub_topic_id
}

resource "google_pubsub_subscription" "transactions_sub" {
  name  = var.pubsub_subscription_id
  topic = google_pubsub_topic.transactions.name

  ack_deadline_seconds = 60   # consumer has 60s to ack each message before redelivery
}
```

The publisher writes to the **topic**; the consumer pulls from the **subscription**. The split lets you run multiple consumers later (each with its own subscription) without changing the publisher.

## 3.4 The publisher and the consumer

!!ADD PUB/SUB AT-LEAST-ONCE DIAGRAM HERE — PUBLISHER → TOPIC → SUBSCRIPTION'S RETAINED MESSAGE BACKLOG → CONSUMER PULLS A MESSAGE → CONSUMER ACKS WITHIN `ack_deadline_seconds` → MESSAGE REMOVED. SECOND PATH: CONSUMER FAILS TO ACK IN TIME → MESSAGE REDELIVERED. CALLOUT: "AT-LEAST-ONCE" MEANS DUPLICATES ARE POSSIBLE BY DESIGN!!

**Publisher** ([scripts/publish_transactions.py](scripts/publish_transactions.py)) reads the streaming CSV, publishes each row as a JSON message at the configured rate, and finishes with a **sentinel** message so the consumer knows when the stream is over:

```python
for idx, row in enumerate(rows):
    payload = dict(row)
    payload["__id"] = idx
    publisher.publish(topic_path, data=json.dumps(payload).encode("utf-8"))
    time.sleep(sleep_seconds)

# Sentinel: tells the consumer to flush its buffer and write the report.
sentinel = json.dumps({"__sentinel": "DONE", "total_published": len(rows)}).encode("utf-8")
publisher.publish(topic_path, data=sentinel)
```

**Consumer** ([scripts/score_stream.py](scripts/score_stream.py)) is a Python pull loop, not Spark Structured Streaming. The reason is pragmatic: the native Structured-Streaming Pub/Sub source needs Dataproc or a Maven jar dance, while a `google-cloud-pubsub` callback into a buffered `spark.createDataFrame` does the same work in one container. The model is still a Spark `PipelineModel` and scoring still goes through Spark DataFrame APIs.

The Pub/Sub callback drops payloads into a shared buffer, with the sentinel triggering shutdown:

```python
def callback(message):
    payload = json.loads(message.data.decode("utf-8"))
    if payload.get("__sentinel") == "DONE":
        with lock:
            state["sentinel_total"] = int(payload.get("total_published", 0))
            state["done"] = True
        message.ack()
        return
    with lock:
        state["buffer"].append(payload)
        state["total_messages"] += 1
    message.ack()
```

And the main loop drains the buffer in batches, scores them through the saved model, and collects flagged frauds:

```python
if buffered >= 50 or (buffered > 0 and time.time() - last_drain > 1.0) or (done and buffered > 0):
    with lock:
        batch = state["buffer"]
        state["buffer"] = []
    df      = spark.createDataFrame(batch_rows, schema=schema)
    scored  = model.transform(df)
    flagged = scored.filter(scored.prediction == 1.0).collect()
```

**At-least-once delivery caveat.** Pub/Sub guarantees *at-least-once* — if the consumer's `message.ack()` is delayed past `ack_deadline_seconds`, the same message is redelivered. That's why the sample report in §3.5 contains six identical fraud entries: it's one underlying transaction, redelivered six times, not six distinct frauds. Production deduplication keys on `__id` (or `Time` + `Amount`) before reporting.

## 3.5 Running it and the sample report

```bash
cd infra/terraform/stream
terraform init
terraform apply        # type 'yes' — creates VPC/NAT/IAP + Pub/Sub + stream VM (~3–5 min total)

# Watch the consumer score messages live
gcloud compute ssh fraud-stream-vm --zone europe-west2-a --tunnel-through-iap \
  --command 'sudo tail -f /var/log/startup.log'

# When the run finishes, read the report straight from GCS
gcloud storage cat gs://<bucket>/reports/fraud_report.json | jq
```

**Sample numbers from a real run** of the deployed pipeline:

| Metric | Value |
|---|---|
| Messages published | 3,000 + 1 sentinel |
| Messages seen by consumer | 3,000 (no loss) |
| Frauds flagged (raw count) | 6 |
| Frauds flagged (deduped on `Time`) | 1 |
| Elapsed seconds | 114.4 |
| Training AUC-ROC / AUC-PR | 0.9645 / 0.7149 |

The six-vs-one gap is the at-least-once redelivery from §3.4 in action.

## 3.6 Tearing everything down

The cleanest finish is the one-shot teardown script, which destroys the stream stack, the train stack (already gone if you followed §2.10), and the data stack — in the right order — and then verifies that no `fraud-*` resources remain:

```bash
# Linux/Mac
bash scripts/destroy_everything.sh

# Windows (PowerShell)
.\scripts\destroy_everything.ps1
```

If the four verification commands at the bottom each print `Listed 0 items.`, you're at zero cloud state and billing has stopped.

# Recap

## 4.2 Applying this to a real fraud-detection workload

The cardholder-transaction pipeline this tutorial models is a faithful skeleton of how a real card-issuer or payment-processor handles fraud detection at scale:

- The **publisher** in §3.4 stands in for the issuer's authorization stream — every card-present and card-not-present transaction posts to a topic in milliseconds.
- The **consumer** scoring loop is what an inference service would do: pull a batch, run features through the saved `PipelineModel`, and emit fraud signals downstream (in production, those signals flow to a decision engine that can decline the transaction in real time).
- The **GCS-stored fraud report** is the audit trail required by financial regulators (PCI DSS, PSD2 SCA evidence). The `Time` + `Amount` + feature columns make every decision reproducible.
- The **separation of train and stream** stacks mirrors a realistic ML lifecycle: a data-science team retrains weekly on a beefy GPU cluster (the train stack), pushes a new versioned image (`fraud-scoring:v6`, `v7`, ...) to a registry, and a serving team rolls it out to a long-running inference fleet (the stream stack) without redeploying any infrastructure.
- The **at-least-once Pub/Sub semantics** are not a bug — they're a deliberate trade. A real fraud system would rather see the same transaction twice (one extra DB lookup) than miss one. Deduplication moves to the decision layer.

The same architectural pattern transfers cleanly to other commercial domains: ad-click fraud, network intrusion detection, telemetry anomaly scoring, or any workload where a model is trained in batch and applied to a continuous event stream.